<a href="https://colab.research.google.com/github/shiragg1/ml-final-snns/blob/main/ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Import libraries

In [ ]:
! pip install snntorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 7.2 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import snntorch as snn
from snntorch import surrogate
import os
import time

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### CNN layers

In [ ]:
class CNNEncoder(nn.Module):
    """
    Extracts features from CIFAR images.
    Progressive feature maps: 3 -> 32 -> 64 -> 128
    """
    def __init__(self):
        super().__init__()

        # Conv block 1: 32x32x3 -> 16x16x32
        self.conv1 = nn.Conv2d(
            in_channels=3,      # RGB input
            out_channels=32,    # 32 feature maps
            kernel_size=3,      # 3x3 filters
            padding=1           # Same padding
        )
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)  # Halves spatial dimensions

        # Conv block 2: 16x16x32 -> 8x8x64
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Conv block 3: 8x8x64 -> 4x4x128
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Output: 4 * 4 * 128 = 2048 values

    def forward(self, x):
        # Block 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)

        # Block 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)

        # Block 3
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)

        # Flatten: (batch, 128, 4, 4) -> (batch, 2048)
        x = x.view(x.size(0), -1)

        return x

### CNN feature map --> spike train function

In [ ]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate


class StraightThroughBernoulli(torch.autograd.Function):
    """Needed for spike generation from CNN features."""
    @staticmethod
    def forward(ctx, probabilities):
        return torch.bernoulli(probabilities)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output  # Pass gradient through unchanged


class SpikeGenerator(nn.Module):
    """Converts CNN features to spike trains with gradient support."""
    def __init__(self, input_dim, num_neurons, num_timesteps):
        super().__init__()
        self.num_timesteps = num_timesteps
        self.proj = nn.Linear(input_dim, num_neurons)

    def forward(self, x):
        probs = torch.sigmoid(self.proj(x))

        spikes = []
        for _ in range(self.num_timesteps):
            spike = StraightThroughBernoulli.apply(probs)  # STE
            spikes.append(spike)

        return torch.stack(spikes, dim=1), probs

### SNN + CNN

In [ ]:

class HybridCNNSNN(nn.Module):
    def __init__(self, num_timesteps=20, num_classes=10, num_snn_neurons=512):
        super().__init__()

        self.num_timesteps = num_timesteps

        # CNN
        self.cnn = CNNEncoder()
        cnn_output_dim = 128 * 4 * 4

        # Spike generator (uses our STE)
        self.num_snn_neurons = num_snn_neurons
        self.spike_gen = SpikeGenerator(cnn_output_dim, num_snn_neurons, num_timesteps)

        # SNN layers (use snnTorch's surrogate gradients)
        spike_grad = surrogate.fast_sigmoid(slope=25)

        self.fc1 = nn.Linear(num_snn_neurons, 256)
        self.lif1 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=True)

        self.fc2 = nn.Linear(256, 128)
        self.lif2 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=True)

        self.fc3 = nn.Linear(128, num_classes)
        self.lif3 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=True)

    def forward(self, x):
        # CNN
        features = self.cnn(x)

        # Spike generation (STE handles gradient)
        input_spikes, spike_probs = self.spike_gen(features)

        # Reset LIF membrane potentials
        self.lif1.reset_mem()
        self.lif2.reset_mem()
        self.lif3.reset_mem()

        # Process through SNN (snnTorch handles gradients)
        output_spikes = []
        for t in range(self.num_timesteps):
            x_t = input_spikes[:, t, :]

            spk1 = self.lif1(self.fc1(x_t))
            spk2 = self.lif2(self.fc2(spk1))
            spk3 = self.lif3(self.fc3(spk2))

            output_spikes.append(spk3)

        output_spikes = torch.stack(output_spikes, dim=1)
        logits = output_spikes.sum(dim=1) / self.num_timesteps

        return logits, spike_probs

### CNN Only

In [ ]:
class CNNOnly(nn.Module):
    """
    Standard CNN classifier for CIFAR-10.
    Same CNN encoder as hybrid, but with FC layers instead of SNN.
    """
    def __init__(self, num_classes=10):
        super().__init__()

        # Same CNN encoder
        self.cnn = CNNEncoder()
        cnn_output_dim = 128 * 4 * 4  # 2048

        # Standard FC classifier (replaces SNN)
        self.classifier = nn.Sequential(
            nn.Linear(cnn_output_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.cnn(x)
        logits = self.classifier(features)
        return logits

### SNN Only

In [ ]:
class SNNOnly(nn.Module):
    """
    Pure SNN classifier for CIFAR-10.
    Converts raw pixels to spikes, then processes with SNN.
    """
    def __init__(self, num_timesteps=20, num_classes=10):
        super().__init__()

        self.num_timesteps = num_timesteps

        # Input: flattened CIFAR image
        input_dim = 32 * 32 * 3  # 3072

        # Spike generator for raw pixels
        self.spike_gen = SpikeGenerator(
            input_dim=input_dim,
            num_neurons=512,
            num_timesteps=num_timesteps
        )

        # SNN layers
        spike_grad = surrogate.fast_sigmoid(slope=25)

        self.fc1 = nn.Linear(512, 256)
        self.lif1 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=True)

        self.fc2 = nn.Linear(256, 128)
        self.lif2 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=True)

        self.fc3 = nn.Linear(128, num_classes)
        self.lif3 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=True)

    def forward(self, x):
        batch_size = x.size(0)

        # Flatten image: (batch, 3, 32, 32) -> (batch, 3072)
        x_flat = x.view(batch_size, -1)

        # Generate spike trains from raw pixels
        input_spikes, spike_probs = self.spike_gen(x_flat)

        # Reset membranes
        self.lif1.reset_mem()
        self.lif2.reset_mem()
        self.lif3.reset_mem()

        # Process through SNN
        output_spikes = []
        for t in range(self.num_timesteps):
            x_t = input_spikes[:, t, :]

            spk1 = self.lif1(self.fc1(x_t))
            spk2 = self.lif2(self.fc2(spk1))
            spk3 = self.lif3(self.fc3(spk2))

            output_spikes.append(spk3)

        output_spikes = torch.stack(output_spikes, dim=1)
        logits = output_spikes.sum(dim=1) / self.num_timesteps

        return logits, spike_probs

### Load data

In [ ]:
def get_data_loaders(batch_size=128, num_workers=2):
    """
    Returns CIFAR-10 train and test data loaders with augmentation.
    """
    # Training transforms with augmentation
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616)
        )
    ])

    # Test transforms (no augmentation)
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.4914, 0.4822, 0.4465),
            std=(0.2470, 0.2435, 0.2616)
        )
    ])

    # Load datasets
    train_dataset = torchvision.datasets.CIFAR10(
        root='./data',
        train=True,
        download=True,
        transform=train_transform
    )

    test_dataset = torchvision.datasets.CIFAR10(
        root='./data',
        train=False,
        download=True,
        transform=test_transform
    )

    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_loader, test_loader

### Training functions

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, epoch, device):
    """Train for one epoch."""
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()

        output = model(images)

        # Handle models that return (logits, spike_probs) vs just logits
        if isinstance(output, tuple):
            logits, spike_probs = output
        else:
            logits = output

        # Compute loss
        loss = criterion(logits, labels)

        # Backward pass
        loss.backward()

        # Gradient clipping (helps stability with surrogate gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Track statistics
        running_loss += loss.item()
        _, predicted = logits.max(dim=1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Print progress
        if (batch_idx + 1) % 100 == 0:
            avg_loss = running_loss / (batch_idx + 1)
            accuracy = 100.0 * correct / total
            print(f"  Batch {batch_idx + 1}/{len(train_loader)}: "
                  f"Loss = {avg_loss:.4f}, Acc = {accuracy:.2f}%")

    epoch_time = time.time() - start_time
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc, epoch_time


def evaluate(model, test_loader, criterion, device):
    """Evaluate model on test set."""
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    # Track spike statistics (if available)
    all_spike_probs = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            # Get model output
            output = model(images)

            # Handle models that return (logits, spike_probs) vs just logits
            if isinstance(output, tuple):
                logits, spike_probs = output
                if spike_probs is not None:
                    all_spike_probs.append(spike_probs.cpu())
            else:
                logits = output

            loss = criterion(logits, labels)

            running_loss += loss.item()
            _, predicted = logits.max(dim=1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    # Compute statistics
    test_loss = running_loss / len(test_loader)
    test_acc = 100.0 * correct / total

    # Spike probability statistics (if available)
    if all_spike_probs:
        all_spike_probs = torch.cat(all_spike_probs, dim=0)
        mean_spike_prob = all_spike_probs.mean().item()
        std_spike_prob = all_spike_probs.std().item()
    else:
        mean_spike_prob = None
        std_spike_prob = None

    return test_loss, test_acc, mean_spike_prob, std_spike_prob

### Training loop

In [ ]:
def train(
    model,
    model_name="model",
    num_epochs=100,
    batch_size=128,
    learning_rate=1e-3,
    save_dir='./checkpoints'
):
    """
    Main training function - works with any model.

    Args:
        model: The model to train (CNNOnly, SNNOnly, or HybridCNNSNN)
        model_name: String identifier for saving checkpoints
        num_epochs: Number of training epochs
        batch_size: Batch size for data loaders
        learning_rate: Initial learning rate
        save_dir: Directory to save checkpoints
    """
    print("=" * 60)
    print(f"Training: {model_name}")
    print("=" * 60)

    # Create save directory
    os.makedirs(save_dir, exist_ok=True)

    # Data loaders
    print("\nLoading data...")
    train_loader, test_loader = get_data_loaders(batch_size=batch_size)
    print(f"Training samples: {len(train_loader.dataset)}")
    print(f"Test samples: {len(test_loader.dataset)}")

    # Move model to device
    print("\nInitializing model...")
    model = model.to(device)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Loss function
    criterion = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=1e-6
    )

    # Training history
    history = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': [],
        'learning_rate': [],
        'spike_prob_mean': [],
        'spike_prob_std': []
    }

    best_test_acc = 0.0

    print("\nStarting training...")
    print("-" * 60)

    for epoch in range(num_epochs):
        current_lr = optimizer.param_groups[0]['lr']
        print(f"\nEpoch {epoch + 1}/{num_epochs} (LR: {current_lr:.6f})")

        # Train
        train_loss, train_acc, epoch_time = train_one_epoch(
            model, train_loader, criterion, optimizer, epoch, device
        )

        # Evaluate
        test_loss, test_acc, spike_mean, spike_std = evaluate(
            model, test_loader, criterion, device
        )

        # Update scheduler
        scheduler.step()

        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['learning_rate'].append(current_lr)
        history['spike_prob_mean'].append(spike_mean)
        history['spike_prob_std'].append(spike_std)

        # Print epoch summary
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Test Loss:  {test_loss:.4f}, Test Acc:  {test_acc:.2f}%")
        if spike_mean is not None:
            print(f"  Spike Prob: {spike_mean:.3f} ± {spike_std:.3f}")
        print(f"  Time: {epoch_time:.1f}s")

        # Save best model
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'test_acc': test_acc,
                'history': history
            }, os.path.join(save_dir, f'best_{model_name}.pt'))
            print(f"  *** New best model saved (Acc: {test_acc:.2f}%) ***")

        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'history': history
            }, os.path.join(save_dir, f'{model_name}_epoch_{epoch + 1}.pt'))

    print("\n" + "=" * 60)
    print("Training complete!")
    print(f"Best test accuracy: {best_test_acc:.2f}%")
    print("=" * 60)

    return model, history

In [ ]:
cnn_model = CNNOnly(num_classes=10)
cnn_model, cnn_history = train(
    model=cnn_model,
    model_name="cnn_only",
    num_epochs=100,
    batch_size=128,
    learning_rate=1e-3,
    save_dir='./checkpoints'
)

Training: cnn_only

Loading data...


100%|██████████| 170M/170M [00:03<00:00, 48.5MB/s]


Training samples: 50000
Test samples: 10000

Initializing model...
Total parameters: 1,276,682
Trainable parameters: 1,276,682

Starting training...
------------------------------------------------------------

Epoch 1/100 (LR: 0.001000)
  Batch 100/391: Loss = 1.9207, Acc = 27.99%
  Batch 200/391: Loss = 1.7820, Acc = 33.18%
  Batch 300/391: Loss = 1.6971, Acc = 36.57%
  Train Loss: 1.6460, Train Acc: 38.71%
  Test Loss:  1.2955, Test Acc:  53.40%
  Time: 20.7s
  *** New best model saved (Acc: 53.40%) ***

Epoch 2/100 (LR: 0.001000)
  Batch 100/391: Loss = 1.3878, Acc = 49.28%
  Batch 200/391: Loss = 1.3541, Acc = 50.83%
  Batch 300/391: Loss = 1.3296, Acc = 51.91%
  Train Loss: 1.3127, Train Acc: 52.63%
  Test Loss:  1.0995, Test Acc:  59.66%
  Time: 17.0s
  *** New best model saved (Acc: 59.66%) ***

Epoch 3/100 (LR: 0.000999)
  Batch 100/391: Loss = 1.2060, Acc = 56.49%
  Batch 200/391: Loss = 1.1839, Acc = 57.54%
  Batch 300/391: Loss = 1.1819, Acc = 57.60%
  Train Loss: 1.1750, T

In [ ]:
snn_model = SNNOnly(num_timesteps=20, num_classes=10)
snn_model, snn_history = train(
    model=snn_model,
    model_name="snn_only",
    num_epochs=100,
    batch_size=128,
    learning_rate=1e-3,
    save_dir='./checkpoints'
)

Training: snn_only

Loading data...


100%|██████████| 170M/170M [00:06<00:00, 28.3MB/s]


Training samples: 50000
Test samples: 10000

Initializing model...
Total parameters: 1,738,890
Trainable parameters: 1,738,890

Starting training...
------------------------------------------------------------

Epoch 1/100 (LR: 0.001000)
  Batch 100/391: Loss = 2.2439, Acc = 16.92%
  Batch 200/391: Loss = 2.1734, Acc = 19.69%
  Batch 300/391: Loss = 2.1398, Acc = 21.61%
  Train Loss: 2.1194, Train Acc: 22.71%
  Test Loss:  2.0941, Test Acc:  23.83%
  Spike Prob: 0.489 ± 0.471
  Time: 39.0s
  *** New best model saved (Acc: 23.83%) ***

Epoch 2/100 (LR: 0.001000)
  Batch 100/391: Loss = 2.0438, Acc = 26.61%
  Batch 200/391: Loss = 2.0402, Acc = 27.00%
  Batch 300/391: Loss = 2.0403, Acc = 27.05%
  Train Loss: 2.0374, Train Acc: 27.14%
  Test Loss:  2.0350, Test Acc:  27.48%
  Spike Prob: 0.488 ± 0.478
  Time: 38.2s
  *** New best model saved (Acc: 27.48%) ***

Epoch 3/100 (LR: 0.000999)
  Batch 100/391: Loss = 2.0225, Acc = 28.12%
  Batch 200/391: Loss = 2.0255, Acc = 27.91%
  Batch 300/

In [ ]:
hybrid_model = HybridCNNSNN(num_timesteps=20, num_classes=10, num_snn_neurons=512)
hybrid_model, hybrid_history = train(
    model=hybrid_model,
    model_name="hybrid_cnn_snn",
    num_epochs=100,
    batch_size=128,
    learning_rate=1e-3,
    save_dir='./checkpoints'
)

Hybrid CNN-SNN Training on CIFAR-10

Loading data...
Training samples: 50000
Test samples: 10000

Initializing model...
Total parameters: 1,308,298
Trainable parameters: 1,308,298

Starting training...
------------------------------------------------------------

Epoch 1/100 (LR: 0.001000)
  Batch 100/391: Loss = 2.1627, Acc = 17.62%
  Batch 200/391: Loss = 2.1063, Acc = 18.54%
  Batch 300/391: Loss = 2.0732, Acc = 20.10%
  Train Loss: 2.0553, Train Acc: 20.88%
  Test Loss:  1.9728, Test Acc:  26.36%
  Spike Prob: 0.303 ± 0.441
  Time: 38.5s
  *** New best model saved (Acc: 26.36%) ***

Epoch 2/100 (LR: 0.001000)
  Batch 100/391: Loss = 1.9752, Acc = 25.88%
  Batch 200/391: Loss = 1.9721, Acc = 26.42%
  Batch 300/391: Loss = 1.9684, Acc = 26.90%
  Train Loss: 1.9666, Train Acc: 27.08%
  Test Loss:  1.9554, Test Acc:  26.60%
  Spike Prob: 0.303 ± 0.444
  Time: 36.3s
  *** New best model saved (Acc: 26.60%) ***

Epoch 3/100 (LR: 0.000999)
  Batch 100/391: Loss = 1.9483, Acc = 28.62%
  Ba

(HybridCNNSNN(
   (cnn): CNNEncoder(
     (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
     (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
     (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
   )
   (spike_gen): SpikeGenerator(
     (proj): Linear(in_features=2048, out_features=512, bias=True)
   )
   (fc1): Linear(in_features=512, out_features=256, bias=True)
   (lif